# CRISP-DM: Donor Impact Allocation

**Analytics goal:** Estimate **which program areas** donations tend to support and **what share** flows to priorities (e.g. education)—to power **transparent donor impact** storytelling and internal targeting.

| CRISP-DM phase | Where it appears |
|----------------|------------------|
| **1. Business understanding** | Next sections (problem framing, KPI mapping) |
| **2. Data understanding** | Donor/donation snapshot, audit |
| **3. Data preparation** | `src.features` |
| **4. Modeling** | Top-area classifier + share explanation |
| **5. Evaluation** | Metrics + business interpretation |
| **6. Deployment** | Donor profile / impact panel |

### 1) Business understanding (impact & allocation narrative)

**Organizational context:** Donors increasingly ask *“where does my money go?”* Lighthouse Sanctuary must connect gifts to **mission programs** (education, safe housing, reintegration, etc.) in a way that is **honest**, **understandable**, and **aligned with fundraising messaging**—without implying false precision.

**Business objectives**
- Provide **credible, donor-facing estimates** of programmatic emphasis (e.g. likely top allocation area, education share) for stewardship and campaigns.
- Help **donor relations** tailor stories: which impact themes resonate for similar past gifts.

**Stakeholders and decisions**
| Stakeholder | Decision enabled |
|-------------|------------------|
| **Donor relations / communications** | Impact copy, annual report callouts, personalized summaries |
| **Fundraising ops** | Segmenting donors by inferred interest themes (for testing, not stereotyping) |
| **Leadership** | Aggregate narrative on program funding mix (with aggregation, not individual overclaim) |

**Measurable success criteria (model + program)**
- Model: reasonable **top-program classification** quality and **share fit** (e.g. macro-F1, R²) as in KPI mapping; non-trivial lift vs. baseline.
- Program: **consistent, stable** estimates across periods where business expects stability; **guardrails** on overclaiming “your gift built X” when uncertainty is high.

**Constraints**
- **Not causal proof:** Historical allocation patterns reflect **accounting and operations**, not counterfactual “your dollar caused exactly this outcome.”
- **Ethics & trust:** Avoid manipulative precision; offer ranges or qualitative bands where appropriate.
- **Equity:** Messaging should not stereotype donors; use segments responsibly.

**Costs of errors**
- Wrong **top program** → misleading thank-you or campaign creative.
- Wrong **share** → mis-set donor expectations and reputational risk.

**Use of outputs:** **Narrative support** and internal triage—paired with finance/program verification for major public claims.

In [ ]:
import os, sys
from pathlib import Path


def _find_ml_pipelines_root() -> Path:
    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        mp = base / "ml-pipelines"
        if mp.is_dir() and (mp / "requirements.txt").is_file():
            return mp
        if base.name == "ml-pipelines" and (base / "requirements.txt").is_file():
            return base
    raise RuntimeError(
        "Could not find ml-pipelines/requirements.txt. Open the Lighthouse-Sanctuary repo "
        "or the ml-pipelines folder, then use Run All."
    )


_boot_path = _find_ml_pipelines_root() / "pipeline_common" / "notebook_bootstrap.py"
_ns = globals()
_ns["__file__"] = str(_boot_path)
exec(compile(_boot_path.read_text(encoding="utf-8"), str(_boot_path), "exec"), _ns)

import pandas as pd

from src.db import load_env, build_engine
from src.features import build_frame, split_xy
from src.modeling import train_predictive_top_area, train_explanatory_share, donor_impact_output


## Problem Framing
- Predictive objective: predict top allocation program area for a donation.
- Explanatory objective: estimate which features are associated with higher education allocation share.
- Stakeholder: donor relations and fundraising operations teams.

## Business KPI mapping (operational measures)

Quantifies the **business objectives** in Business understanding for model and program review.

- Decision enabled: donor impact estimate display and targeting narrative.
- Primary KPI: top-program prediction quality (macro F1) and allocation-share fit (R2).
- Guardrails: avoid overclaiming causal outcomes; maintain estimate stability across periods.
- Success criteria: non-trivial predictive lift and interpretable driver consistency.

In [ ]:
load_env('.env')
engine = build_engine(os.getenv('DB_PROFILE', 'local'))
df = build_frame(engine)
X, y_top, y_share_education = split_xy(df)
print('Rows:', len(df), '| Distinct donors:', df['supporter_id'].nunique())
df.head()

In [ ]:
# Data Understanding Audit
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
display(missing_pct.head(10).to_frame('missing_pct'))
audit = {
    'rows': len(df),
    'program_area_classes': int(y_top.nunique()),
    'zero_donation_value_rows': int((df['donation_value'] <= 0).sum()),
}
print('Audit summary:', audit)
print('Feature rationale: donation/supporter context explains likely allocation patterns observed in history.')

In [ ]:
pred_metrics, best_top_model = train_predictive_top_area(X, y_top)
print('Predictive metrics (top program area):')
display(pred_metrics)

In [ ]:
expl_metrics, linear_share_model, rf_share_model, coef_df = train_explanatory_share(X, y_share_education)
print('Share model metrics:')
display(expl_metrics)
print('Top positive explanatory drivers (education share):')
display(coef_df.head(12))
print('Top negative explanatory drivers:')
display(coef_df.tail(12).sort_values('coefficient'))

In [ ]:
impact_out = donor_impact_output(df, best_top_model, linear_share_model)
print('Donor impact output sample:')
display(impact_out.head(20))

## Evaluation in Business Terms
- Misclassifying top area can mislead donor messaging.
- Over/under-estimating allocation shares can distort expectation setting.
- Use outputs as impact estimates based on historical patterns, not guaranteed causal outcomes.

## Operationalization Policy + Monitoring
- Prediction policy: display top-program estimate + education-share range.
- Action bands: high education-share estimates for education-focused messaging.
- Retraining cadence: monthly or when monitored quality drops.
- References:
  - `ml-pipelines/integration/pipeline_registry.yaml`
  - `ml-pipelines/integration/monitoring_spec.md`
  - `ml-pipelines/integration/README.md`

## Deployment Notes
Integrate as donor profile panel: input donation context, output estimated allocation profile and uncertainty range.